In [ ]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


In [ ]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    String payload = """
        exec('''%s''')
        print('%s')
        """.formatted(code.replace("'", "\\'"), marker);

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}


In [ ]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_3'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


In [ ]:
%%jshell

runPy("""
from sklearn.neighbors import LocalOutlierFactor
import numpy as np # Ensure numpy is imported for np.where
import pandas as pd # Import pandas to load DataFrame

# 3. Density based outlier detection using - (Chosen) Local outlier Factor (LOF)
threshold = -6

# Ensure 'data' is defined from the DataFrame 'df'
# Load df if not already loaded
DATASET_PATH = "./housing.csv"
# Download the dataset if it's not present
import os
if not os.path.exists(DATASET_PATH):
    print(f"Downloading {DATASET_PATH}...")
os.system('gdown 1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP print(f"{DATASET_PATH} downloaded.") df = pd.read_csv(DATASET_PATH) data = df["SalePrice"]');

lof = LocalOutlierFactor(n_neighbors=10)
lof.fit(data.to_frame())
lof_scores = lof.negative_outlier_factor_
outlier_indices = np.where(lof_scores < threshold)[0]
outlier_points = data[outlier_indices]

outlier_points
""");


This code snippet performs Density-Based Outlier Detection using the Local Outlier Factor (LOF) algorithm on your SalePrice data. Here's a breakdown of what each part does:

# 3. Density based outlier detection using - (Chosen) Local outlier Factor (LOF):

This is a comment indicating the method chosen for outlier detection.
threshold = -6:

This line sets a threshold value. In LOF, smaller (more negative) negative_outlier_factor_ values indicate a higher likelihood of being an outlier. This specific threshold of -6 will be used to determine which points are considered outliers.
lof = LocalOutlierFactor(n_neighbors=10):

This initializes the LocalOutlierFactor model from scikit-learn. The n_neighbors=10 parameter specifies that for each point, the algorithm will consider its 10 nearest neighbors to calculate its local density and compare it with its neighbors' densities.
lof.fit(data.to_frame()):

This step fits the LOF model to your data. data.to_frame() converts your SalePrice Series into a DataFrame, which is the expected input format for the fit method. During this fitting, the model computes the local reachability density for each sample.
lof_scores = lof.negative_outlier_factor_:

After fitting, the negative_outlier_factor_ attribute contains the opposite of the LOF score for each data point. A value close to -1 indicates that the sample is similar to its neighbors (not an outlier), while significantly lower (more negative) values indicate a lower local density than its neighbors, thus making it an outlier.
outlier_indices = np.where(lof_scores < threshold)[0]:

This line identifies the indices of the data points that are considered outliers. It selects all points whose negative_outlier_factor_ is less than the threshold of -6. np.where() returns a tuple of arrays, and [0] extracts the array of indices.
outlier_points = data[outlier_indices]:

Using the identified outlier_indices, this line extracts the actual SalePrice values from your original data Series that correspond to these outlier points.
outlier_points:

This line will display the pandas Series containing the SalePrice values that have been identified as outliers by the LOF algorithm.

In [ ]:
%%jshell

runPy("""
# Analysis and removal techniques of outliers from dataset
#
# 1. Visual analysis of the Univariate data to observe extreme values in the data
#    a. Distplot (Histplot)
#    b. Boxplots/Whisker plots
#    c. Scatter plot
#
# 2. Distance based outlier detection - K-NN based outlier detection
#
# 3.
#
#`n
#
#
""");


In [ ]:
%%jshell

runPy("""
# Download the dataset file to local storage (from publicly shared drive file)
# Original source: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
# (The rest of the notebook expects the original housing.csv to be present in the current directory)
os.system('gdown 1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP');
""");


In [ ]:
%%jshell

runPy("""
# Import required libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import LocalOutlierFactor
""");


In [ ]:
%%jshell

runPy("""
# Load the dataset
import pandas as pd
DATASET_PATH = "./housing.csv"
df = pd.read_csv(DATASET_PATH)
df
""");


In [ ]:
%%jshell

runPy("""
# Select the data for for analysis/outlier detection
data = df["SalePrice"]
data
""");


In [ ]:
%%jshell

runPy("""
# a. Distplot using Histplot
sns.histplot(data, kde=True)
pass
""");


In [ ]:
%%jshell

runPy("""
# b. Boxplots/ Whisker plots
sns.boxplot(data)
pass
""");


In [ ]:
%%jshell

runPy("""
# c. Scatter plot
#
# from matplotlib.ticker import MaxNLocator # to ensure the graph show int on y axis (can b)
# fig, ax = plt.subplots()
# ax.yaxis.set_major_locator(MaxNLocator(integer=True))

sns.scatterplot(data.value_counts())
# plt.xlabel('SalePrice')
plt.ylabel('Frequency')
pass
""");


In [ ]:
%%jshell

runPy("""
# 2. Distance based outlier detection - K-NN based outlier detection
""");



Certainly! This code snippet performs K-Nearest Neighbors (k-NN) based outlier detection on your SalePrice data. Here's a breakdown of what each part does:

model = NearestNeighbors(n_neighbors=10): This line initializes a NearestNeighbors model from scikit-learn. It's configured to find the 10 nearest neighbors for each data point.model.fit(data.to_frame()): The model is trained using your data (which is the 'SalePrice' column). data.to_frame() converts the pandas Series into a DataFrame, which is the expected input format for the fit method.

distances, _ = model.kneighbors(data.to_frame()): This step calculates the distances from each data point to its 10 nearest neighbors. distances will be an array where each row corresponds to a data point, and columns contain the distances to its neighbors, sorted from closest to furthest.

The _ is used to ignore the indices of the neighbors, as we only need the distances here.

outlier_score = distances[:, -1]: For each data point, its outlier score is determined by the distance to its k-th (in this case, 10th) nearest neighbor. If a point is far from its 10th neighbor, it's considered more isolated and thus more likely to be an outlier. This line extracts that 10th distance for every point.

threshold = np.percentile(outlier_score, 95): A threshold for identifying outliers is set. Here, the 95th percentile of all calculated outlier scores is chosen. This means that data points with an outlier score higher than this value will be considered outliers.

outliers = data[outlier_score > threshold]: Finally, this line identifies and extracts the actual data points from your original data Series that have an outlier_score greater than the calculated threshold. These are your detected outliers.

outliers: This line will display the pandas Series containing the SalePrice values that have been identified as outliers by the k-NN algorithm.

In [ ]:
%%jshell

runPy("""
print(df.dtypes)
print(data)
""");


In [ ]:
%%jshell

runPy("""
model = NearestNeighbors(n_neighbors=10)
model.fit(data.to_frame())

# Find distances to k nearest neighbors for each point
distances, _ = model.kneighbors(data.to_frame())

# Compute outlier score as the distance to the kth nearest neighbor
outlier_score = distances[:, -1]

# Set threshold for outlier detection
threshold = np.percentile(outlier_score, 95)

# Identify outliers
outliers = data[outlier_score > threshold]
outliers
""");


In [ ]:
%%jshell

runPy("""
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure with two subplots
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Histogram of outliers (from k-NN)
sns.histplot(outliers, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Outlier SalePrices (k-NN)')
axes[0].set_xlabel('SalePrice')
axes[0].set_ylabel('Frequency')

# Boxplot of outliers (from k-NN)
sns.boxplot(x=outliers, ax=axes[1])
axes[1].set_title('Boxplot of Outlier SalePrices (k-NN)')
axes[1].set_xlabel('SalePrice')

plt.tight_layout()
plt.show()
""");


In [ ]:
%%jshell

runPy("""
# 3. Density based outlier detection using - (Chosen) Local outlier Factor (LOF)
threshold = -6

lof = LocalOutlierFactor(n_neighbors=10)
lof.fit(data.to_frame())
lof_scores = lof.negative_outlier_factor_
outlier_indices = np.where(lof_scores < threshold)[0]
outlier_points = data[outlier_indices]

outlier_points
""");


In [ ]:
%%jshell

runPy("""
# 4) Remove outliers using trimming and capping (Winsorization)
trimming_threshold_percent = 10

lower_bound = np.percentile(data, trimming_threshold_percent)
upper_bound = np.percentile(data, 100 - trimming_threshold_percent)

# a) Trimming (based on percentile)
trimmed_data = data[(data > lower_bound) & (data < upper_bound)]
trimmed_data
""");


In [ ]:
%%jshell

runPy("""
# Distplot on trimmed data
sns.histplot(trimmed_data, kde=True, stat="density", ec='w')
pass
""");


In [ ]:
%%jshell

runPy("""
# Boxplots/ Whisker plots on trimmed data
sns.boxplot(trimmed_data)
pass
""");


In [ ]:
%%jshell

runPy("""
# b) Winsorization (based on percentile)
capping_threshold_percent = 10

lower_bound = np.percentile(data, capping_threshold_percent)
upper_bound = np.percentile(data, 100 - capping_threshold_percent)

capped_data = np.clip(data, lower_bound, upper_bound)
capped_data
""");


In [ ]:
%%jshell

runPy("""
# Distplot on capped data
sns.histplot(capped_data, kde=True, stat="density", ec='w')
pass
""");


In [ ]:
%%jshell

runPy("""
# Boxplots/ Whisker plots on capped data
sns.boxplot(capped_data)
pass
""");


In [ ]:
%%jshell

runPy("""

import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# 1. Prepare Data
# Reshape to 2D array for scikit-learn compatibility: (n_samples, n_features)
X = np.array([[1], [2], [3], [4], [5]])  # Study hours
y = np.array([2, 3, 5, 6, 8])             # Exam scores

# 2. Initialize and Fit the Model
model = LinearRegression()
model.fit(X, y)

# 3. Model Parameters
print(f"Slope (beta_1): {model.coef_[0]:.2f}")
print(f"Intercept (beta_0): {model.intercept_:.2f}")

# 4. Predict for new data (e.g., 6 hours)
new_hours = np.array([[6]])
predicted_score = model.predict(new_hours)
print(f"Predicted score for 6 hours: {predicted_score[0]:.2f}")

# 5. Visualizing Results
plt.scatter(X, y, color='blue', label='Actual Data')
plt.plot(X, model.predict(X), color='red', label='Linear Fit Line')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.title('Linear Regression Model')
plt.legend()
plt.show()
""");


In [ ]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");
